# 02 - Retrieval evaluation

Compare retrieval strategies on the ground-truth question set from notebook 01.

Metrics: Hit@k (does the gold study id appear in top-k?) and MRR (mean
reciprocal rank). We report Hit@1, Hit@3, Hit@5, MRR for each strategy.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from ingest import load_documents, load_embeddings, load_text_index
from search import TextSearcher, VectorSearcher, HybridSearcher, RerankedSearcher, RewritingSearcher, QueryRewriter
from llmclient import get_llm_client

docs = load_documents()
text_index = load_text_index()
emb = load_embeddings()

with open('ground_truth.json') as f:
    ground_truth = json.load(f)
print(len(ground_truth), 'questions,', len(docs), 'studies')

In [ ]:
def evaluate(searcher, gt, k=5):
    hits = {1: 0, 3: 0, 5: 0}
    mrr = 0.0
    for row in gt:
        results = searcher.search(row['question'], num_results=k)
        ids = [r.get('study_id') for r in results]
        target = row['study_id']
        for kk in (1, 3, 5):
            if target in ids[:kk]:
                hits[kk] += 1
        if target in ids:
            mrr += 1.0 / (ids.index(target) + 1)
    n = len(gt) or 1
    return {'Hit@1': hits[1]/n, 'Hit@3': hits[3]/n, 'Hit@5': hits[5]/n, 'MRR': mrr/n}

text = TextSearcher(text_index, docs)
vec = VectorSearcher(emb, docs)
hyb = HybridSearcher(text, vec)
rer = RerankedSearcher(hyb)
client = get_llm_client()
rew = RewritingSearcher(rer, QueryRewriter(client))

results = {}
for name, s in [
    ('text', text),
    ('vector', vec),
    ('hybrid', hyb),
    ('hybrid+rerank', rer),
    ('hybrid+rerank+rewrite', rew),
]:
    print(name, '...')
    results[name] = evaluate(s, ground_truth)

import pandas as pd
df = pd.DataFrame(results).T
df

In [ ]:
df.to_csv('retrieval_eval.csv')
best = df['MRR'].idxmax()
print('best strategy by MRR:', best)